In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
PROJECT_PATH = "/content/drive/MyDrive/VictorianGPT"
clean_folder = f"{PROJECT_PATH}/cleaned"
dialogue_folder = f"{PROJECT_PATH}/dialogues"

# Ensure the dialogues folder exists
os.makedirs(dialogue_folder, exist_ok=True)
print("Drive mounted and directories verified.")

Mounted at /content/drive
Drive mounted and directories verified.


In [2]:
texts = []

for file in os.listdir(clean_folder):
    if file.endswith(".txt"):
        with open(f"{clean_folder}/{file}", 'r', encoding="utf8") as f:
            texts.append(f.read())

combined_text = "\n".join(texts)

print(f"Total corpus size: {len(combined_text)} characters")
print(f"Books loaded: {[f for f in os.listdir(clean_folder) if f.endswith('.txt')]}")

Total corpus size: 3274638 characters
Books loaded: ['jane_eyre.txt', 'dorian_gray.txt', 'dracula.txt', 'great_expectations.txt']


In [4]:
import re

# We use both standard (" ") and smart (“ ”) quotes.
# re.DOTALL allows the .* to match across line breaks (\n).
raw_dialogues = re.findall(r'["“](.*?)["”]', combined_text, re.DOTALL)

dialogues = []
for d in raw_dialogues:
    # Replace internal newlines with spaces so the quote is one continuous string
    clean_quote = d.replace('\n', ' ').strip()

    # Clean up any weird multi-spacing
    clean_quote = re.sub(r'\s+', ' ', clean_quote)

    # Filter out very short fragments or empty strings
    if len(clean_quote) > 5:
        dialogues.append(clean_quote)

print(f"Total individual quotes extracted: {len(dialogues)}")

# Let's peek at the first 3 to make sure they look right
if dialogues:
    print("\nSample Quotes:")
    for i in range(min(3, len(dialogues))):
        print(f"{i+1}. {dialogues[i]}")
else:
    print("Still 0. Check if combined_text is empty!")

Total individual quotes extracted: 9994

Sample Quotes:
1. Jane Eyre
2. Jane Eyre:
3. Vanity Fair


In [5]:
# Let's look at a slice from the middle (e.g., index 5000 to 5005)
print("Peeking at the middle of the dataset:")
for i in range(5000, 5005):
    if i < len(dialogues):
        print(f"Quote {i}: {dialogues[i]}")
        print("-" * 30)

Peeking at the middle of the dataset:
Quote 5000: I am a little changed already.
------------------------------
Quote 5001: You cannot change to me, Dorian,
------------------------------
Quote 5002: You and I will always be friends.
------------------------------
Quote 5003: Yet you poisoned me with a book once. I should not forgive that. Harry, promise me that you will never lend that book to any one. It does harm.
------------------------------
Quote 5004: My dear boy, you are really beginning to moralize. You will soon be going about like the converted, and the revivalist, warning people against all the sins of which you have grown tired. You are much too delightful to do that. Besides, it is no use. You and I are what we are, and will be what we will be. As for being poisoned by a book, there is no such thing as that. Art has no influence upon action. It annihilates the desire to act. It is superbly sterile. The books that the world calls immoral are books that show the world its 

In [6]:
pairs = []

# Map consecutive quotes as conversation pairs (Input -> Output)
for i in range(len(dialogues)-1):
    input_text = dialogues[i]
    output_text = dialogues[i+1]

    # Enforce a minimum length threshold (e.g., > 15 chars)
    # This prevents the model from learning to respond to long prompts with a simple "Yes." or "I see."
    if len(input_text) > 15 and len(output_text) > 15:
        pairs.append({
            "input": input_text,
            "output": output_text
        })

print(f"Total authentic conversational pairs created: {len(pairs)}")

Total authentic conversational pairs created: 7096


In [7]:
import json

save_path = f"{dialogue_folder}/victorian_dataset.json"

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(pairs, f, indent=4)

print(f"Production dataset successfully saved to: {save_path}")

Production dataset successfully saved to: /content/drive/MyDrive/VictorianGPT/dialogues/victorian_dataset.json


In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_PATH="/content/drive/MyDrive/VictorianGPT"

import os

os.makedirs(PROJECT_PATH,exist_ok=True)

print(PROJECT_PATH)

/content/drive/MyDrive/VictorianGPT


In [3]:
!pip install -q transformers accelerate sentencepiece datasets

In [4]:
from transformers import pipeline

generator=pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto"
)

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded


In [5]:
base_prompts=[

"I am tired",
"I am hungry",
"I failed my exam",
"I feel lonely",
"I had a terrible day",
"I got rejected from an internship",
"I feel happy",
"I miss my friends",
"I am anxious",
"I want to sleep",
"I feel stressed",
"I am excited",
"I lost my phone",
"I want advice",
"I am confused",

]

print(len(base_prompts))

15


In [6]:
expanded=[]

emotions=[
"sad","happy","lonely","anxious",
"stressed","worried","confused",
"excited","angry"
]

situations=[
"after exams",
"after an interview",
"after bad news",
"while studying",
"after losing something",
"during college",
"after an argument",
"after a long day"
]

actions=[
"I feel",
"I became",
"I am",
"I suddenly feel",
"I think I am",
"I am feeling"
]

all_inputs=[]

for a in actions:
    for e in emotions:
        for s in situations:
            all_inputs.append(
                f"{a} {e} {s}"
            )

print(len(all_inputs))

432


In [7]:
import os

path="/content/drive/MyDrive/VictorianGPT/dialogues/victorian_dataset.json"

if os.path.exists(path):
    os.remove(path)

print("Old dataset removed")

Old dataset removed


In [8]:
from tqdm import tqdm

dataset=[]

for text in tqdm(all_inputs):

    prompt=f"""
You are converting modern English into natural Victorian English.

Rules:

- Speak like an educated person from late 19th-century England
- Avoid Shakespeare completely
- Never use:
thou,thee,thy,dost,hast,methinks,yea
- Keep replies concise (1–3 sentences)
- Do not invent fictional situations
- Do not pretend to physically accompany the user
- Never explain your writing choices
- Never say:
"feel free to modify"
- Reply naturally as a chatbot

Modern sentence:
{text}

Victorian response:
"""



    output=generator(
    prompt,
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)


    response=output[0]["generated_text"]

    response=response.split(
        "Victorian version:"
    )[-1].strip()

    dataset.append({
        "input":text,
        "output":response
    })

  0%|          | 0/432 [00:00<?, ?it/s]Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  2%|▏         | 10/432 [00:17<11:25,  1.62s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 432/4

In [9]:
import json

save_path=f"{PROJECT_PATH}/victorian_dataset.json"

with open(
    save_path,
    "w"
) as f:

    json.dump(
        dataset,
        f,
        indent=4
    )

print("saved")
print(save_path)

saved
/content/drive/MyDrive/VictorianGPT/victorian_dataset.json
